In [ ]:
from astral import LocationInfo
from astral.sun import sun
import pandas as pd

city = LocationInfo(
    "Paris",
    "France",
    "Europe/Paris",
    48.8566,
    2.3522
)

def trajectory_period(df):

    if len(df) == 0:
        return "unknown"

    t = pd.to_datetime(
        df["datetime"].iloc[0]
    )

    if t.tzinfo is None:
        t = t.tz_localize("UTC")

    t_local = t.tz_convert("Europe/Paris")

    s = sun(
        city.observer,
        date=t_local.date(),
        tzinfo="Europe/Paris"
    )

    if (
        t_local < s["dawn"] or
        t_local > s["dusk"]
    ):
        return "night"

    return "day"

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import osmnx as ox
import folium

from leuvenmapmatching.map.inmem import InMemMap
from leuvenmapmatching.matcher.distance import DistanceMatcher

# =========================================================
# PARAMETERS
# =========================================================

GPX_FOLDER = "paris_traces_walking"
N_TRAJ = 20

# =========================================================
# LOAD GRAPH
# =========================================================

G = ox.load_graphml("paris_walk.graphml")

# =========================================================
# BUILD LEUVEN GRAPH
# =========================================================

map_con = InMemMap(
    "london_walk",
    use_latlon=True,
    use_rtree=False,
    index_edges=True
)

for node, data in G.nodes(data=True):
    map_con.add_node(
        node,
        (data["y"], data["x"])
    )

for u, v in G.edges():
    map_con.add_edge(u, v)

matcher = DistanceMatcher(
    map_con,
    max_dist=30,
    max_dist_init=30,
    obs_noise=10,
    obs_noise_ne=30,
    dist_noise=15,
    non_emitting_length_factor=0.5,
    max_lattice_width=10
)

# =========================================================
# GPX LOADER
# =========================================================

NS = {
    "gpx": "http://www.topografix.com/GPX/1/0"
}

def load_gpx(path):

    tree = ET.parse(path)
    root = tree.getroot()

    pts = []

    for pt in root.findall(".//gpx:trkpt", NS):

        pts.append([
            float(pt.attrib["lat"]),
            float(pt.attrib["lon"])
        ])

    return pd.DataFrame(
        pts,
        columns=["lat","lon"]
    )

# =========================================================
# STATES -> COORDS
# =========================================================

def states_to_coords(states):

    coords = []

    for s in states:

        try:

            if isinstance(s, tuple):
                node = s[0]
            else:
                node = s

            coords.append(
                (
                    G.nodes[node]["y"],
                    G.nodes[node]["x"]
                )
            )

        except:
            pass

    return coords

# =========================================================
# MAP
# =========================================================
def save_maps(df, forward, backward, idx):

    # ==========================
    # FORWARD MAP
    # ==========================

    m = folium.Map(
        location=[
            df.lat.mean(),
            df.lon.mean()
        ],
        zoom_start=15
    )

    folium.PolyLine(
        list(zip(df.lat, df.lon)),
        color="red",
        weight=3,
        tooltip="Raw GPS"
    ).add_to(m)

    if len(forward) > 1:

        folium.PolyLine(
            forward,
            color="blue",
            weight=5,
            tooltip="Forward HMM"
        ).add_to(m)

    legend = """
    <div style="
    position: fixed;
    bottom:50px;
    left:50px;
    width:170px;
    background:white;
    border:2px solid grey;
    z-index:9999;
    padding:10px;">
    <b>Forward HMM</b><br>
    <span style='color:red;'>━━</span> Raw GPS<br>
    <span style='color:blue;'>━━</span> HMM
    </div>
    """

    m.get_root().html.add_child(
        folium.Element(legend)
    )

    m.save(
        f"forward_{idx}.html"
    )

    # ==========================
    # REVERSE MAP
    # ==========================

    m = folium.Map(
        location=[
            df.lat.mean(),
            df.lon.mean()
        ],
        zoom_start=15
    )

    folium.PolyLine(
        list(zip(df.lat, df.lon)),
        color="red",
        weight=3,
        tooltip="Raw GPS"
    ).add_to(m)

    if len(backward) > 1:

        folium.PolyLine(
            backward,
            color="green",
            weight=5,
            tooltip="Reverse HMM"
        ).add_to(m)

    legend = """
    <div style="
    position: fixed;
    bottom:50px;
    left:50px;
    width:170px;
    background:white;
    border:2px solid grey;
    z-index:9999;
    padding:10px;">
    <b>Reverse HMM</b><br>
    <span style='color:red;'>━━</span> Raw GPS<br>
    <span style='color:green;'>━━</span> HMM
    </div>
    """

    m.get_root().html.add_child(
        folium.Element(legend)
    )

    m.save(
        f"reverse_{idx}.html"
    )
def compare_map(df, forward, backward, outfile):

    m = folium.Map(
        location=[
            df.lat.mean(),
            df.lon.mean()
        ],
        zoom_start=15
    )

    # Raw GPS
    folium.PolyLine(
        list(zip(df.lat, df.lon)),
        color="red",
        weight=3,
        tooltip="Raw"
    ).add_to(m)

    # Forward HMM
    if len(forward) > 1:

        folium.PolyLine(
            forward,
            color="blue",
            weight=5,
            tooltip="Forward"
        ).add_to(m)

    # Reverse HMM
    if len(backward) > 1:

        folium.PolyLine(
            backward,
            color="green",
            weight=5,
            tooltip="Reverse"
        ).add_to(m)

    legend = """
    <div style="
    position: fixed;
    bottom:50px;
    left:50px;
    width:180px;
    background:white;
    border:2px solid grey;
    z-index:9999;
    padding:10px;
    ">
    <b>Legend</b><br>
    <span style='color:red;'>━━</span> Raw GPS<br>
    <span style='color:blue;'>━━</span> Forward HMM<br>
    <span style='color:green;'>━━</span> Reverse HMM
    </div>
    """

    m.get_root().html.add_child(
        folium.Element(legend)
    )

    m.save(outfile)



In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import networkx as nx
from pyproj import Geod

# ---------------------------------------
# PARAMETERS
# ---------------------------------------
GPX_FOLDER='paris_traces_walking'
N_TRAJ = None      # None = all trajectories

geod = Geod(ellps="WGS84")

# ---------------------------------------
# RAW LENGTH
# ---------------------------------------

def raw_length(df):

    if len(df) < 2:
        return 0

    length = 0

    for i in range(len(df)-1):

        _, _, d = geod.inv(
            df.lon.iloc[i],
            df.lat.iloc[i],
            df.lon.iloc[i+1],
            df.lat.iloc[i+1]
        )

        length += d

    return length


# ---------------------------------------
# MATCHED LENGTH
# ---------------------------------------

def matched_length(nodes):

    if len(nodes) < 2:
        return 0

    length = 0

    for u, v in zip(nodes[:-1], nodes[1:]):

        if G.has_edge(u, v):

            data = G.get_edge_data(u, v)

            edge = min(
                data.values(),
                key=lambda x: x.get("length", 0)
            )

            length += edge.get("length", 0)

        else:

            try:

                length += nx.shortest_path_length(
                    G,
                    u,
                    v,
                    weight="length"
                )

            except:

                pass

    return length


# ---------------------------------------
# STATES -> NODES
# ---------------------------------------

def states_to_nodes(states):

    nodes = []

    for s in states:

        if isinstance(s, tuple):
            nodes.append(s[0])
        else:
            nodes.append(s)

    return nodes


# ---------------------------------------
# RESULTS
# ---------------------------------------

results = []

files = sorted([
    f for f in os.listdir(GPX_FOLDER)
    if f.endswith(".gpx")
])

if N_TRAJ is not None:
    files = files[:N_TRAJ]

# ---------------------------------------
# LOOP
# ---------------------------------------

for file in files:

    print(file)

    df = load_gpx(
        os.path.join(
            GPX_FOLDER,
            file
        )
    )

    if len(df) < 5:
        continue

    pts = list(zip(df.lat, df.lon))

    raw_len = raw_length(df)

    period = trajectory_period(df)

    # -----------------------
    # Forward
    # -----------------------

    try:

        states_f, _ = matcher.match(pts)

    except:

        states_f = []

    # -----------------------
    # Reverse
    # -----------------------

    try:

        states_r, _ = matcher.match(
            pts[::-1]
        )

        states_r = states_r[::-1]

    except:

        states_r = []

    nodes_f = states_to_nodes(states_f)
    nodes_r = states_to_nodes(states_r)

    len_f = matched_length(nodes_f)
    len_r = matched_length(nodes_r)

    results.append({

        "file": file,

        "period": period,

        "gps_points": len(df),

        "raw_length_m": raw_len,

        "forward_success": len(nodes_f) > 0,

        "reverse_success": len(nodes_r) > 0,

        "forward_states": len(nodes_f),

        "reverse_states": len(nodes_r),

        "forward_coverage":
            len(nodes_f) / len(df),

        "reverse_coverage":
            len(nodes_r) / len(df),

        "forward_length_m": len_f,

        "reverse_length_m": len_r,

        "forward_ratio":
            len_f / raw_len if raw_len > 0 else np.nan,

        "reverse_ratio":
            len_r / raw_len if raw_len > 0 else np.nan

    })

# ---------------------------------------
# DATAFRAME
# ---------------------------------------

stats = pd.DataFrame(results)

stats.to_csv(
    "map_matching_statistics.csv",
    index=False
)

print(stats.head())

# ---------------------------------------
# SUMMARY
# ---------------------------------------

print("\n========== SUMMARY ==========")

print("Total traces:", len(stats))

print(
    "Forward success:",
    stats.forward_success.mean()*100
)

print(
    "Reverse success:",
    stats.reverse_success.mean()*100
)

print(
    "Either:",
    (
        stats.forward_success |
        stats.reverse_success
    ).mean()*100
)

print(
    "Both:",
    (
        stats.forward_success &
        stats.reverse_success
    ).mean()*100
)

print("\nDAY / NIGHT")

print(

    stats.groupby("period")[

        [
            "forward_success",
            "reverse_success",
            "forward_coverage",
            "reverse_coverage",
            "forward_ratio",
            "reverse_ratio"

        ]

    ].mean()

)
